# Kaggle score and strategy monitor

This notebook records the public submission history, current leaderboard, replay-harvest metadata, and public notebook categories for the Pokémon TCG AI Battle Challenge. It compares observable outcomes with implementation signals to identify evidence-backed strategy patterns.

It does not access private submissions, bypass Kaggle controls, execute competitor code, or claim that correlations prove hidden rules.

In [ ]:
from pathlib import Path
import json, os, re, time
from datetime import datetime, timezone
import pandas as pd

COMPETITION = 'pokemon-tcg-ai-battle'
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT / 'data' / 'score_monitor'
OUT.mkdir(parents=True, exist_ok=True)

token = Path.home() / '.kaggle' / 'access_token'
if token.exists(): os.environ.setdefault('KAGGLE_API_TOKEN', token.read_text().strip())
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print('Authenticated; output:', OUT)

In [ ]:
def field(obj, *names, default=None):
    for name in names:
        value = getattr(obj, name, None)
        if value not in (None, ''): return value
    return default

def number(value):
    try: return float(value)
    except (TypeError, ValueError): return None

submissions = []
for item in api.competition_submissions(COMPETITION) or []:
    submissions.append({
        'submission_id': field(item, 'ref', 'id', '_ref'),
        'date': str(field(item, 'date', '_date', default='')),
        'description': str(field(item, 'description', '_description', default='')),
        'status': str(field(item, 'status', '_status', default='')),
        'public_score': number(field(item, 'public_score', '_public_score')),
        'private_score': number(field(item, 'private_score', '_private_score')),
        'error': field(item, 'error_description', '_error_description'),
        'file_name': field(item, 'file_name', '_file_name'),
    })

leaderboard = []
for rank, item in enumerate(api.competition_leaderboard_view(COMPETITION) or [], 1):
    leaderboard.append({
        'rank': rank,
        'team_name': str(field(item, 'team_name', '_team_name', default='')),
        'score': number(field(item, 'score', '_score')),
        'team_id': field(item, 'team_id', '_team_id'),
    })

snapshot = {'captured_at': datetime.now(timezone.utc).isoformat(), 'competition': COMPETITION,
            'submissions': submissions, 'leaderboard': leaderboard}
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
(OUT / f'{stamp}.json').write_text(json.dumps(snapshot, indent=2, ensure_ascii=False))
(OUT / 'latest.json').write_text(json.dumps(snapshot, indent=2, ensure_ascii=False))
print('Submissions:', len(submissions), '| leaderboard rows:', len(leaderboard))
display(pd.DataFrame(submissions))

In [ ]:
# Build a time series from all local snapshots.
snapshots = []
for path in sorted(OUT.glob('*.json')):
    if path.name == 'latest.json': continue
    try: snapshots.append(json.loads(path.read_text()))
    except json.JSONDecodeError: pass
history = []
for snap in snapshots:
    for row in snap.get('submissions', []):
        history.append({'captured_at': snap.get('captured_at'), **row})
hist = pd.DataFrame(history)
if len(hist):
    hist['captured_at'] = pd.to_datetime(hist['captured_at'])
    hist = hist.sort_values(['submission_id', 'captured_at'])
    hist['score_change'] = hist.groupby('submission_id')['public_score'].diff()
    display(hist[['captured_at','submission_id','description','status','public_score','score_change']])
    print('Score changes by submission:')
    display(hist.dropna(subset=['score_change']).sort_values('score_change', ascending=False))
else:
    print('Run this notebook again after later submissions to create score-change history.')

In [ ]:
# Compare public code signals from the scout with observable notebook votes.
manifest_path = ROOT / 'kaggle_code' / 'public_scout' / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    code_rows = []
    for row in manifest.get('kernels', []):
        signals = row.get('code_signals', {})
        code_rows.append({
            'author': row.get('author'), 'title': row.get('title'),
            'votes': row.get('votes'), 'research_priority': row.get('research_priority'),
            'categories': ', '.join(signals.get('categories', [])),
            'mcts_rl': signals.get('has_mcts'),
            'search_api': signals.get('has_search_api'),
            'agent_code': signals.get('has_submission_agent'),
            'url': row.get('public_url'),
        })
    display(pd.DataFrame(code_rows).sort_values('research_priority', ascending=False))
else:
    print('Run tools/scout_public_code.py first to populate public code signals.')

## Interpretation guardrails

- A score change is not automatically caused by a code change; matchmaking, opponent pool, randomness, and evaluation timing also matter.
- Public notebook votes and leaderboard rank are proxies, not official Gold/Silver/Bronze labels.
- Use repeated snapshots and matched local evaluations before treating a pattern as a rule.
- Keep downloaded public code in research-only storage and review it manually before adapting ideas.